# Proceso de automatizacion contable

In [1]:
# importar librerias a usar
import pandas as pd
from clase_reportes_new import ReportClassNew
import numpy as np
import tkinter as tk
from tkinter import filedialog
import json
import re
import os
import fitz  # PyMuPDF
rc = ReportClassNew()


In [2]:
pdfs = [r"d:\Downloads\ICA.pdf", r"d:\Downloads\FTE.pdf"]

In [5]:
import fitz  # PyMuPDF

list_path = [r"d:\Downloads\ICA.pdf", r"d:\Downloads\FTE.pdf"]


In [ ]:
def separar_pdfs_estratégico(self, ruta_pdf_input, carpeta_salida, find_texto="NIT:", nombre_base="ICA", nit_carpeta=True, list_path=None, df=None,
                            enumerar=0):
        if list_path:
            # Crear documento vacío
            pdf_unido = fitz.open()

            # Recorrer los PDFs y agregarlos
            for pdf in pdfs:
                doc = fitz.open(pdf)
                pdf_unido.insert_pdf(doc)
             
        
        else :
            if not os.path.exists(carpeta_salida):
                os.makedirs(carpeta_salida)

                doc = fitz.open(ruta_pdf_input)
                total_paginas = len(doc)
                print(f"🚀 Procesando {nombre_base} | {total_paginas} páginas...")

        procesados = 0
        
        
            


        for i in range(total_paginas):
            pagina = doc[i]
            texto = pagina.get_text()
            
            # Extracción de NITs [cite: 7, 29, 50, 72]
            nits_encontrados = re.findall(rf"{find_texto}\s*(\d+-?\d*)", texto)
            
            # El NIT del tercero es el segundo en la estructura de nuestros certificados [cite: 12, 34, 56, 77]
            if len(nits_encontrados) >= 2:
                nit_cliente = nits_encontrados[1].replace("-", "").strip()
            else:
                nit_cliente = "REVISAR_MANUAL"

            # Definir ubicación de la carpeta del cliente
            ruta_destino_final = carpeta_salida
            if nit_carpeta:
                ruta_destino_final = os.path.join(carpeta_salida, nit_cliente)
                if not os.path.exists(ruta_destino_final):
                    os.makedirs(ruta_destino_final)

            # --- Lógica de Contador Inteligente (Verifica archivos existentes) ---
            contador = 1
            
            if enumerar > 1:
                nombre_archivo = f"{nit_cliente}_{nombre_base}_{enumerar+contador}.pdf"
            else:
                nombre_archivo = f"{nit_cliente}_{nombre_base}.pdf"

            ruta_final = os.path.join(ruta_destino_final, nombre_archivo)
            
            # Si ya existe un archivo con ese nombre (ej. procesaste ICA antes), añade sufijo
            while os.path.exists(ruta_final):
                contador += 1
                nombre_archivo = f"{nit_cliente}_{nombre_base}_{contador}.pdf"
                ruta_final = os.path.join(ruta_destino_final, nombre_archivo)

            # Guardado del PDF
            nuevo_doc = fitz.open()
            nuevo_doc.insert_pdf(doc, from_page=i, to_page=i)
            nuevo_doc.save(ruta_final)
            nuevo_doc.close()

            procesados += 1
            if procesados % 10 == 0 or procesados == total_paginas:
                print(f"📊 {nombre_base}: {procesados}/{total_paginas} completadas...")

    


        doc.close()
        print(f"✅ Finalizado: {nombre_base} organizado en {carpeta_salida}.\n")

